# ML debiasing of ERA5-Land vs IFS-HRES

**Goal.** Learn a correction that maps ERA5-Land archive values toward the IFS-HRES
historical-forecast values, using a gradient-boosted tree (XGBoost / CatBoost /
LightGBM — try whichever). The model captures the *systematic* (orographic,
geographic, seasonal) component of the archive-vs-forecast gap so we can debias
the served ERA5-Land temperatures.

## Target

```
y = era5_land_value - hres_value      # the bias we want to predict & subtract
```
(per cell, per day, for `tmax_C` to start — extend to `tmin_C` later.)

## Features

| feature              | source                                              |
|----------------------|-----------------------------------------------------|
| `era5_land`          | the archive value itself (the thing being corrected)|
| `elevation`          | ERA5-Land model orography at the cell (CDS `z`)     |
| `lat`, `lon`         | `cells.csv`                                         |
| `dist_to_hres_km`    | haversine(cell center, HRES grid point)             |
| `elev_diff_m`        | `elevation` (ERA5-Land) − `hres_elevation`          |
| `cos_doy`            | `cos(2*pi*doy/365)` seasonal phase                  |
| `sin_doy`            | `sin(2*pi*doy/365)`  (add the pair — one cos is ambiguous) |

> Note added a `sin_doy` companion: a lone `cos(doy)` can't distinguish spring
> from autumn (same value), so the seasonal cycle is only half-resolved without
> the sine. Drop it if you truly want just cos.

## Data provenance (important — don't mix DEMs)

- **ERA5-Land temps** came from the EarthDataHub ARCO **Zarr** store
  (`reanalysis-era5-land-no-antartica-v0.zarr`), *not* Open-Meteo.
- **Cell elevation** = ERA5-Land model orography on the **same 0.1° grid** as the
  temps, from the CDS `reanalysis-era5-land` static geopotential (NOT a generic
  DEM, NOT Open-Meteo GLO-90, NOT the 0.25° ARCO ERA5). See §2 — already built to
  `data/cell_elevation.csv`.
- **HRES forecast** values + `hres_elevation` come from the bias-study puller
  (`pull_hres_all.py` → R2 `hres-forecast/`, mirrored locally under
  `data/hres-forecast/`). The HRES grid-point elevation is recorded in
  `.hres_progress.json`.

See `FINDINGS.md` for the archive bugs (F1 coastal-snap, F2) already surfaced —
make sure those cells are either fixed or excluded before training.

In [ ]:
# Env. The base interpreter has no xarray/sklearn — install into the notebook env.
# %pip install xarray zarr requests pandas numpy scikit-learn xgboost catboost lightgbm cfgrib
import json, gzip, math
from pathlib import Path
import numpy as np
import pandas as pd

BIAS_DIR = Path('.').resolve()                      # scripts/bias_study
REPO     = BIAS_DIR.parents[1]                       # repo root
CELLS_CSV   = REPO / 'data' / 'cells.csv'
HRES_DIR    = BIAS_DIR / 'data' / 'hres-forecast'
HRES_LEDGER = HRES_DIR / '.hres_progress.json'
print('repo:', REPO)
print('cells:', CELLS_CSV.exists(), '| hres dir:', HRES_DIR.exists(), '| ledger:', HRES_LEDGER.exists())

## 1. Load the cell list and the HRES ledger

In [ ]:
cells = pd.read_csv(CELLS_CSV)            # cell_id, lat, lon, population, tile_*, name
cells = cells[['cell_id', 'lat', 'lon', 'name']].copy()

# HRES ledger: done['<lat>_<lon>'] -> {name, rows, hres_lat, hres_lon, hres_elevation}
ledger = json.load(open(HRES_LEDGER))['done']
hres_meta = pd.DataFrame([
    {'key': k, 'hres_lat': v['hres_lat'], 'hres_lon': v['hres_lon'],
     'hres_elevation': v['hres_elevation'], 'hres_rows': v['rows']}
    for k, v in ledger.items()
])
# the ledger key is the ERA5-Land cell lat_lon, e.g. '23.8_90.4'
cells['key'] = cells['lat'].map(lambda x: f'{x:g}') + '_' + cells['lon'].map(lambda x: f'{x:g}')
cells = cells.merge(hres_meta, on='key', how='inner')   # only cells we have HRES for
print(len(cells), 'cells with HRES forecast')
cells.head()

## 2. ERA5-Land cell elevation (model orography)

Elevation = ERA5-Land **static surface geopotential** sampled at each cell center.

**This was already built** → `data/cell_elevation.csv` (`cell_id, elevation`),
10,181 cells, 0 NaN. Source + method (run once, no need to repeat):

- The EarthDataHub hourly Zarr (where the *temps* came from) does **not** carry a
  static `z`, and at the time was 429-rate-limited. Google ARCO has `z` but only
  at 0.25° plain ERA5 — wrong grid. So elevation was pulled from the **CDS**
  `reanalysis-era5-land` `geopotential` invariant field, which is the **same
  0.1° grid as the temps** (canonical orography).
- CDS request (one-shot static field; pick any date — it's time-invariant):
  ```python
  c.retrieve('reanalysis-era5-land',
      {'variable': 'geopotential', 'year': '2020', 'month': '01',
       'day': '01', 'time': '00:00', 'data_format': 'grib'},
      'data/era5land_geopotential.grib')   # ~6 MB zip wrapping geo.grib
  ```
- Sampled bilinearly at cell centers (lon mapped `%360` to match the 0..360 grid),
  ocean-NaN coastal fallback to nearest land. Geometric height via the **exact**
  formula `h = z·Re/(g·Re − z)` (Re=6371000, g=9.80665) — differs from naive
  `z/g` by ≤2.7 m even in the Himalaya, but used correctly anyway.

The build cell below regenerates `cell_elevation.csv` from `geo.grib` if the cache
is missing; normally it just loads the cache. Validated against known city
elevations (Mexico City 2183 m, Bogotá 2605 m, Denver 1602 m, …).

In [ ]:
ELEV_CACHE = BIAS_DIR / 'data' / 'cell_elevation.csv'        # cell_id, elevation
GEO_GRIB   = BIAS_DIR / 'data' / 'geo.grib'                  # CDS era5-land z field
G, RE = 9.80665, 6371000.0

if ELEV_CACHE.exists():
    elev = pd.read_csv(ELEV_CACHE)
    print('loaded cached elevation:', len(elev))
else:
    # Regenerate from the CDS geopotential GRIB (see markdown above for the pull).
    import xarray as xr
    z = xr.open_dataset(GEO_GRIB, engine='cfgrib')['z']     # m^2/s^2, lat 90..-90, lon 0..360
    src = pd.read_csv(CELLS_CSV)
    lat = src['lat'].values
    lon = src['lon'].values % 360.0                          # match 0..360 grid
    lin = z.interp(latitude=('cell', lat), longitude=('cell', lon), method='linear').values
    nn  = z.interp(latitude=('cell', lat), longitude=('cell', lon), method='nearest').values
    gpot = np.where(np.isnan(lin), nn, lin)                  # coastal -> nearest land
    h = gpot * RE / (G * RE - gpot)                          # exact geometric height
    elev = src[['cell_id']].copy(); elev['elevation'] = np.round(h, 1)
    elev.to_csv(ELEV_CACHE, index=False)
    print('built cell_elevation.csv:', len(elev), '| NaN:', int(np.isnan(h).sum()))

elev['elevation'].describe()

In [ ]:
cells = cells.merge(elev, on='cell_id', how='left')
assert cells['elevation'].notna().all(), 'missing elevation for some cells'
cells.head()

## 3. Static per-cell geometry features

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

cells['dist_to_hres_km'] = haversine_km(cells['lat'], cells['lon'],
                                        cells['hres_lat'], cells['hres_lon'])
cells['elev_diff_m'] = cells['elevation'] - cells['hres_elevation']
cells[['cell_id','lat','lon','elevation','hres_elevation','elev_diff_m','dist_to_hres_km']].describe()

## 4. Load HRES forecast time series + join the ERA5-Land archive

HRES files: `data/hres-forecast/hres_<hlat>_<hlon>.csv.gz` with columns
`date, tmax_C, tmin_C, precip_mm, wind_max_ms`. The filename uses the **HRES**
lat/lon (from the ledger), not the cell center.

The matching ERA5-Land archive per cell/day lives in R2 (`era5-land/` prefix) /
the served DB. **TODO:** point `load_era5_land(cell)` at wherever you want to read
the archive from for the study (R2, a local mirror, or re-derive from the Zarr).
It must return a frame indexed by `date` with a `tmax_C` column on the LOCAL solar
day (the archive is bucketed by local day — see project notes).

In [ ]:
def hres_path(row):
    return HRES_DIR / f"hres_{row['hres_lat']:g}_{row['hres_lon']:g}.csv.gz"

def load_hres(row):
    df = pd.read_csv(hres_path(row), parse_dates=['date'])
    return df[['date', 'tmax_C', 'tmin_C']].rename(
        columns={'tmax_C': 'hres_tmax', 'tmin_C': 'hres_tmin'})

def load_era5_land(row):
    """TODO: return DataFrame[date, era5_tmax, era5_tmin] for this cell.
    Read from the R2 era5-land archive (local-day buckets). Placeholder below."""
    raise NotImplementedError('wire to the ERA5-Land archive (R2 / local mirror)')

# sanity check on one cell before the full build
r0 = cells.iloc[0]
print('HRES file:', hres_path(r0).exists())
load_hres(r0).head()

In [ ]:
# Build the long training frame: one row per (cell, day).
frames = []
for _, row in cells.iterrows():
    try:
        h = load_hres(row)
        e = load_era5_land(row)
    except FileNotFoundError:
        continue
    m = h.merge(e, on='date', how='inner')
    if m.empty:
        continue
    for col in ['cell_id','lat','lon','elevation','hres_elevation',
                'elev_diff_m','dist_to_hres_km']:
        m[col] = row[col]
    frames.append(m)

df = pd.concat(frames, ignore_index=True)
doy = df['date'].dt.dayofyear
df['cos_doy'] = np.cos(2*np.pi*doy/365.0)
df['sin_doy'] = np.sin(2*np.pi*doy/365.0)

# target + the era5 feature (start with tmax)
df['era5_land'] = df['era5_tmax']
df['bias'] = df['era5_tmax'] - df['hres_tmax']   # y
print(df.shape)
df[['cell_id','date','era5_land','hres_tmax','bias']].head()

## 5. Train

**Split by cell, not by row** — otherwise the model leaks per-cell offsets via
neighboring days and the test score is optimistic. Hold out whole cells (and
ideally whole regions) so the debiaser is evaluated on cells it has never seen.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

FEATURES = ['era5_land', 'elevation', 'lat', 'lon',
            'dist_to_hres_km', 'elev_diff_m', 'cos_doy', 'sin_doy']
TARGET = 'bias'

data = df.dropna(subset=FEATURES + [TARGET])
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr, te = next(gss.split(data, groups=data['cell_id']))
train, test = data.iloc[tr], data.iloc[te]
X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]
print('train rows', len(train), '| test rows', len(test),
      '| test cells', test['cell_id'].nunique())

In [ ]:
# Pick your boost. XGBoost shown; CatBoost/LightGBM are drop-in alternatives.
from xgboost import XGBRegressor
model = XGBRegressor(
    n_estimators=600, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=0,
)
model.fit(X_train, y_train)

## 6. Evaluate — did debiasing actually help?

The honest baseline is **doing nothing**: how big is the raw bias? The model is
only worth shipping if the *residual* after subtracting the predicted bias beats
the raw bias on held-out cells.

In [ ]:
from sklearn.metrics import mean_absolute_error
pred = model.predict(X_test)
raw_mae   = mean_absolute_error(y_test, np.zeros_like(y_test))  # |bias| with no correction
resid_mae = mean_absolute_error(y_test, pred)                   # |bias - predicted_bias|
print(f'raw bias MAE      : {raw_mae:.3f} °C')
print(f'residual MAE      : {resid_mae:.3f} °C')
print(f'improvement       : {100*(1-resid_mae/raw_mae):.1f}%')

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nfeature importance:'); print(imp)

In [ ]:
# Debiased archive value = era5_land - predicted_bias. Quick diagnostic plots.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(y_test, pred, s=2, alpha=0.2); ax[0].plot([-10,10],[-10,10],'r--')
ax[0].set(xlabel='true bias', ylabel='predicted bias', title='bias fit')
ax[1].hist(y_test, bins=60, alpha=0.5, label='raw bias')
ax[1].hist(y_test - pred, bins=60, alpha=0.5, label='residual')
ax[1].legend(); ax[1].set_title('bias distribution')
plt.tight_layout()

## Next steps

- Repeat for `tmin_C` (separate model or a multi-output target).
- Region-held-out CV, not just random cell holdout, to test geographic transfer.
- Try CatBoost (handles the geographic features well) and compare.
- Exclude / fix F1+F2 archive-bug cells (`FINDINGS.md`) before trusting the fit.
- If it generalizes, export the model + apply it in the serving path as a
  per-cell/day correction lookup.